# Scrape a JavaScript Page with Selenium

Some pages build their content with JavaScript, so `requests` alone sees an empty page. This notebook runs headless Chrome with Selenium 4 in Colab, waits for the content to load and extracts it. It uses the JavaScript version of the public practice site quotes.toscrape.com.

## How to run
Run all cells in order. Results are saved to `quotes.csv`.

_Rewritten from my original Colab notebook: `Untitled8.ipynb`._

In [ ]:
!apt-get update -qq
!apt-get install -y -qq chromium-driver
!pip install -q selenium pandas

## Start headless Chrome
Selenium 4 can find a driver automatically (Selenium Manager); on Colab we point it at the installed chromedriver if it exists.

In [ ]:
import os
import time
import urllib.robotparser
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=learning-scraper/1.0 (educational project)")

driver_path = "/usr/bin/chromedriver"
service = Service(driver_path) if os.path.exists(driver_path) else Service()
driver = webdriver.Chrome(service=service, options=options)

In [ ]:
START_URL = "https://quotes.toscrape.com/js/"
NUM_PAGES = 3

robots = urllib.robotparser.RobotFileParser("https://quotes.toscrape.com/robots.txt")
robots.read()
print("Allowed:", robots.can_fetch("*", START_URL))

## Visit pages and extract quotes

In [ ]:
rows = []
try:
    for page in range(1, NUM_PAGES + 1):
        url = START_URL if page == 1 else f"{START_URL}page/{page}/"
        driver.get(url)
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.quote"))
        )
        for q in driver.find_elements(By.CSS_SELECTOR, "div.quote"):
            rows.append({
                "text": q.find_element(By.CSS_SELECTOR, "span.text").text,
                "author": q.find_element(By.CSS_SELECTOR, "small.author").text,
            })
        print(f"Page {page}: {len(rows)} quotes so far")
        time.sleep(1.5)  # be polite
finally:
    driver.quit()

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
df.to_csv("quotes.csv", index=False)
df.head()